In [1]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda  path: "/lessons/" in path #, #"02-vector-search/lessons/" in path, #02-vector-search/lessons/07-sqlitesearch-vector.md
)

documents = [file.parse() for file in reader.read()]


In [2]:
# Q1
from embedder import Embedder
import onnxruntime as ort
from sentence_transformers import SentenceTransformer
embed = Embedder()

model = SentenceTransformer("all-MiniLM-L6-v2")

query = "How does approximate nearest neighbor search work?"
v = embed.encode(query)
v1 = v[0]
v1

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

np.float64(-0.020582034180917443)

In [3]:
from gitsource import GithubRepositoryDataReader

reader2 = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "02-vector-search/lessons/07-sqlitesearch-vector.md" in path
)

documents2 = [file.parse() for file in reader2.read()]
v2 = embed.encode(documents2[0]['content'])
v2.dot(v)

np.float64(0.36107008472347096)

In [4]:
# Q3 

from gitsource import chunk_documents
chunks = chunk_documents(documents, size=2000, step=1000)
#chunks[0]#['content']

In [16]:
#from tqdm.auto import tqdm
vectors = []
batch_size = 50
batch = []

for i in range(0, len(chunks), batch_size):
    #batch.extend(str(chunks[j]['content']) for j in range(i, 50))
    batch.extend([
    str(chunks[j]['content']) 
    for j in range(i, min(50, len(chunks)))
])
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)
    
batch_vectors


array([[-0.11451445, -0.0301345 ,  0.00268035, ...,  0.09064332,
        -0.02903816,  0.02549705],
       [ 0.03725683, -0.02299264,  0.03227818, ..., -0.00060152,
        -0.05212052,  0.02472224],
       [-0.04115276,  0.0007708 , -0.04108728, ...,  0.03442788,
        -0.02607915,  0.0048584 ],
       ...,
       [-0.07179876,  0.08688132, -0.01847803, ...,  0.04634552,
         0.00066773,  0.04387844],
       [-0.03219134,  0.0484371 , -0.02562968, ...,  0.08525855,
         0.01026535, -0.01750406],
       [-0.02799191,  0.04995072,  0.04720151, ..., -0.01041199,
         0.08381656,  0.00767035]], shape=(50, 384), dtype=float32)

In [17]:
import numpy as np
X = np.array(vectors)
X.shape

(300, 384)

In [19]:
query = "How does approximate nearest neighbor search work?"
v_query = embed.encode(query)

scores = X.dot(v_query)
#scores = [v_query.dot(X[i]) for i in range(len(X))]

idx = np.argmax(scores)
idx, scores[idx]
documents[idx]

{'content': '# ToyAIKit\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=PQpQOR3Un3w&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nThe handwritten agent loop from the previous lesson is educational but\nrepetitive. Every time you build a new agent, you\'d write the same\nwhile-loop, the same function-call handling, the same message\nmanagement.\n\nToyAIKit wraps this pattern so you can focus on tools, prompts, and\nbehavior. We built it together in a DataTalks.Club workshop a while\nback. It does the same thing as our handwritten loop with less\nboilerplate. If you open its `runners` code, you\'ll find the same\n`while True` loop we wrote by hand.\n\nI use it here on purpose, because I don\'t want to pick a winner among\nthe production frameworks. ToyAIKit is small and easy to read, so when\nsomething breaks you can see exactly what happened. That makes it handy\nfor developing and debugging locally before you go to production.\n\nOne caveat. ToyAIKit is a teaching and exper

In [23]:
print(len(chunks))

295


In [21]:
print(len(X))

300


In [24]:
# Q4
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=["course"])
vindex.fit(X, chunks)


ValueError: Number of vectors must match number of payload documents

In [14]:
query = "What metric do we use to evaluate a search engine?"
query_vector = embed.encode(query)

results = VectorSearch(query_vector, num_results=5)
results

TypeError: VectorSearch.__init__() got an unexpected keyword argument 'num_results'

In [ ]:
results = vindex.search(
    query_vector,
    #filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)